<a href="https://colab.research.google.com/github/chandanar8126/titanic-ml-pluto-academy/blob/main/Titanic_ML_PlutoAcademy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Titanic Survival Prediction — Machine Learning Model
### Pluto Academy AI & ML Internship | Project 02
**Name:** Chandana R
**Dataset:** Titanic Dataset (Kaggle)
**Objective:** Build, train and evaluate 3 machine learning models to predict
survival of Titanic passengers and identify the best performing model.

In [3]:
# importing all libraries needed for this project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules for ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# clean plot styling
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("all libraries loaded successfully")

all libraries loaded successfully


## Step 1 — Loading, Exploring and Preprocessing the Data
First I'll load the dataset, inspect it carefully, handle missing values,
encode categorical variables and split into train/test sets.

In [4]:
# loading the titanic dataset from google drive
# dataset source: https://www.kaggle.com/c/titanic

df = pd.read_csv('/content/drive/MyDrive/train.csv')

# first look at the data
print("Shape of dataset:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape of dataset: (891, 12)

Column Names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# checking missing values in each column
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

print("Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

Missing Values:
          Missing Count  Missing %
Age                 177      19.87
Cabin               687      77.10
Embarked              2       0.22


In [6]:
# basic statistics of the dataset
print("Basic Statistics:")
df.describe()

Basic Statistics:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
# checking survival distribution - this is what we are predicting
survival_counts = df['Survived'].value_counts()

print("Survival Distribution:")
print(f"Did not survive: {survival_counts[0]} passengers")
print(f"Survived: {survival_counts[1]} passengers")
print(f"\nSurvival Rate: {(survival_counts[1]/len(df)*100).round(2)}%")

Survival Distribution:
Did not survive: 549 passengers
Survived: 342 passengers

Survival Rate: 38.38%


## Preprocessing Decisions

Before training models I need to:
1. Drop columns that are not useful for prediction
2. Handle missing values in Age, Cabin and Embarked
3. Encode categorical columns (Sex, Embarked) into numbers
4. Split data into train and test sets (80/20)

In [8]:
# dropping columns that won't help predict survival
# PassengerId - just a serial number, no predictive value
# Name - individual names don't help prediction
# Ticket - ticket numbers are random, not useful
# Cabin - 77% missing, too many gaps to be useful

cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=cols_to_drop)

print("Remaining columns:", df.columns.tolist())
print("Shape after dropping:", df.shape)

Remaining columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Shape after dropping: (891, 8)


In [12]:
# filling missing Age values with median
# using median because age data can be skewed by outliers
df['Age'] = df['Age'].fillna(df['Age'].median())

# filling missing Embarked with mode (most common value)
# only 2 rows missing - safe to fill with most common port
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# confirming no missing values remain
print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nShape:", df.shape)

Missing values after cleaning:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

Shape: (891, 8)


In [10]:
# converting categorical columns to numbers
# ML models only understand numbers, not text

# Sex column: male = 1, female = 0
df['Sex'] = LabelEncoder().fit_transform(df['Sex'])

# Embarked column: S = 2, Q = 1, C = 0
df['Embarked'] = LabelEncoder().fit_transform(df['Embarked'])

print("After encoding:")
print(df[['Sex', 'Embarked']].value_counts())
print("\nSample data:")
df.head()

After encoding:
Sex  Embarked
1    2           441
0    2           205
1    0            95
0    0            73
1    1            41
0    1            36
Name: count, dtype: int64

Sample data:


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [11]:
# separating features (X) and target (y)
# target is what we want to predict - Survived column
X = df.drop(columns=['Survived'])
y = df['Survived']

# splitting into 80% training and 20% testing
# random_state=42 ensures same split every time we run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nFeatures used for prediction:")
print(X.columns.tolist())

Training set size: (712, 7)
Testing set size: (179, 7)

Features used for prediction:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
